# Week 5: Exploring AWS AI Services in SageMaker Notebooks

## Learning Objectives

By the end of this session, you will be able to:
1. **Navigate** the SageMaker notebook environment (Jupyter interface, boto3 client initialization)
2. **Call AWS AI Services** APIs from a notebook (Comprehend, Textract, Rekognition)
3. **Parse JSON responses** and structure data into pandas DataFrames
4. **Save processed results** to S3 for downstream analytics

## Prerequisites

- Completed Weeks 3-4 (Databricks/Spark experience)
- Watched pre-class videos on AWS fundamentals, S3, and SageMaker overview
- AWS Console access with your student credentials

## SageMaker Studio Environment

You're running this notebook in **Amazon SageMaker Studio**, a fully managed IDE for machine learning. Key features:
- Pre-configured Python environment with AWS SDK (boto3)
- IAM role with permissions to call AWS services
- Direct access to S3 buckets without explicit credentials

# Section 0: Environment Setup

In this section, we'll:
1. Import required libraries
2. Initialize AWS service clients (boto3)
3. Verify access to our S3 bucket with datasets

In [ ]:
# =============================================================================
# IMPORTS AND AWS CLIENT INITIALIZATION
# =============================================================================
# Standard libraries
import boto3                    # AWS SDK for Python
import pandas as pd             # Data manipulation
import json                     # JSON parsing for API responses
from datetime import datetime   # Timestamps for output files
import time                     # For API rate limiting

# For displaying images in notebook (optional)
from IPython.display import Image, display
import io

# =============================================================================
# STUDENT IDENTIFICATION
# =============================================================================
# You will be prompted to enter your student number when you run this cell

STUDENT_NAME = input("Enter your student number (e.g., student1): ").strip()

if not STUDENT_NAME:
    raise ValueError("❌ Please enter your student number!")

print(f"✅ Student: {STUDENT_NAME}")

# =============================================================================
# INITIALIZE AWS CLIENTS
# =============================================================================
# SageMaker notebooks have credentials pre-configured via IAM role
# No need to provide access keys - boto3 automatically uses the execution role

session = boto3.Session()
region = session.region_name

# AI Services clients - these are the services we'll use today
comprehend = boto3.client('comprehend', region_name=region)  # NLP service
textract = boto3.client('textract', region_name=region)      # Document OCR
rekognition = boto3.client('rekognition', region_name=region) # Image analysis

# S3 client for data access
s3 = boto3.client('s3', region_name=region)

# Get account ID for bucket naming
sts = boto3.client('sts')
account_id = sts.get_caller_identity()['Account']

# =============================================================================
# DEFINE BUCKET AND PATHS
# =============================================================================
# Our academy S3 bucket follows the naming convention: sagemaker-academy-{account_id}
BUCKET_NAME = f"sagemaker-academy-{account_id}"
DATA_PREFIX = "week5/data"
OUTPUT_PREFIX = f"week5/outputs/{STUDENT_NAME}"  # Each student gets their own folder

print("=" * 60)
print("AWS Environment Setup Complete")
print("=" * 60)
print(f"Student:    {STUDENT_NAME}")
print(f"Region:     {region}")
print(f"Account:    {account_id}")
print(f"S3 Bucket:  {BUCKET_NAME}")
print(f"Data Path:  s3://{BUCKET_NAME}/{DATA_PREFIX}/")
print(f"Output:     s3://{BUCKET_NAME}/{OUTPUT_PREFIX}/")
print("=" * 60)

In [ ]:
# =============================================================================
# VERIFY DATA ACCESS
# =============================================================================
# Before we start processing, let's confirm we can access all our datasets

print("Verifying access to datasets in S3...\n")

# Customer feedback CSV
print("1. Customer Feedback (Text Data for Comprehend):")
response = s3.list_objects_v2(Bucket=BUCKET_NAME, Prefix=f"{DATA_PREFIX}/customer_feedback")
for obj in response.get('Contents', []):
    print(f"   [OK] {obj['Key']}")

# Scanned forms
print("\n2. Scanned Forms (Images for Textract):")
response = s3.list_objects_v2(Bucket=BUCKET_NAME, Prefix=f"{DATA_PREFIX}/scanned_forms/")
form_count = 0
for obj in response.get('Contents', []):
    if obj['Key'].endswith('.png'):
        print(f"   [OK] {obj['Key'].split('/')[-1]}")
        form_count += 1
print(f"   Total: {form_count} form images")

# Product images  
print("\n3. Product Images (for Rekognition):")
response = s3.list_objects_v2(Bucket=BUCKET_NAME, Prefix=f"{DATA_PREFIX}/product_images/")
image_count = 0
for obj in response.get('Contents', []):
    if obj['Key'].endswith('.jpg') or obj['Key'].endswith('.jpeg'):
        print(f"   [OK] {obj['Key'].split('/')[-1]}")
        image_count += 1
print(f"   Total: {image_count} product images")

print("\n" + "=" * 60)
print("All datasets accessible!")
print("=" * 60)

# What Are We Building Today?

## Scenario: Customer Insight Dashboard Data Pipeline

Imagine you're a data scientist at a retail company. Your company receives customer feedback through **three different channels**:

| Channel | Data Type | Example | AWS Service |
|---------|-----------|---------|-------------|
| **Text Reviews** | Emails, survey responses | "Great product but shipping was slow..." | Amazon Comprehend |
| **Scanned Forms** | Paper feedback forms | Customer complaint form (PDF/image) | Amazon Textract |
| **Product Photos** | Images customers send | Photo of damaged item | Amazon Rekognition |

Your task: Build the **data extraction layer** that powers a Customer Insight Dashboard.

## The Pipeline We'll Build

```
                    Customer Feedback
                           |
         +-----------------+-----------------+
         |                 |                 |
    Text Reviews     Scanned Forms     Product Photos
         |                 |                 |
    Comprehend         Textract        Rekognition
         |                 |                 |
    - Sentiment        - OCR Text       - Labels
    - Entities         - Form Fields    - Objects
    - Key Phrases      - Key-Values     - Confidence
         |                 |                 |
         +-----------------+-----------------+
                           |
                    Structured Data
                           |
                      Save to S3
                           |
                    Dashboard / ML
```

## Why This Matters

Instead of manually reading thousands of feedback items, we can:
- **Automatically classify** sentiment (positive/negative/neutral)
- **Extract structured data** from unstructured documents
- **Identify objects** in images for quality control
- **Feed downstream systems** (dashboards, ML models, alerts)

In [ ]:
# =============================================================================
# LOAD CUSTOMER FEEDBACK DATA
# =============================================================================
# This CSV contains 20 customer feedback records that we'll analyze with Comprehend

feedback_key = f"{DATA_PREFIX}/customer_feedback.csv"

# Download from S3 and load into pandas
response = s3.get_object(Bucket=BUCKET_NAME, Key=feedback_key)
feedback_df = pd.read_csv(response['Body'])

print(f"Loaded {len(feedback_df)} customer feedback records\n")
print("Columns:", list(feedback_df.columns))
print("\nFirst 3 records:")
display(feedback_df.head(3))

# Let's look at one feedback text in detail
print("\n" + "=" * 60)
print("Sample Feedback Text:")
print("=" * 60)
print(feedback_df.iloc[0]['feedback_text'])

# Section 1: Amazon Comprehend - Text Analysis

Amazon Comprehend is a **Natural Language Processing (NLP)** service that extracts insights from text without requiring any ML expertise.

## What Can Comprehend Do?

| API | Purpose | Output |
|-----|---------|--------|
| `detect_sentiment()` | Determine emotional tone | POSITIVE, NEGATIVE, NEUTRAL, MIXED + confidence scores |
| `detect_entities()` | Extract named entities | People, organizations, locations, dates, quantities |
| `detect_key_phrases()` | Identify important phrases | Key topics and concepts mentioned |

## API Pattern

All Comprehend APIs follow the same simple pattern:

```python
# Call the API with text and language
response = comprehend.detect_sentiment(
    Text="Your text here...",
    LanguageCode="en"  # English
)

# Response is a dictionary with results
print(response['Sentiment'])  # "POSITIVE"
print(response['SentimentScore'])  # {'Positive': 0.95, 'Negative': 0.01, ...}
```

## Demo: Analyzing Customer Feedback

Let's start by analyzing a single piece of feedback to understand the API responses.

In [ ]:
# =============================================================================
# DEMO: SENTIMENT ANALYSIS ON SINGLE TEXT
# =============================================================================
# Let's analyze the first feedback record to see how Comprehend works

sample_text = feedback_df.iloc[0]['feedback_text']
print("Input Text:")
print(sample_text)
print("\n" + "=" * 60)

# Call Comprehend's detect_sentiment API
sentiment_response = comprehend.detect_sentiment(
    Text=sample_text,
    LanguageCode='en'  # English
)

# The response contains:
# - Sentiment: The overall sentiment label (POSITIVE, NEGATIVE, NEUTRAL, MIXED)
# - SentimentScore: Confidence scores for each sentiment category

print("\nSentiment Analysis Results:")
print(f"  Overall Sentiment: {sentiment_response['Sentiment']}")
print(f"\n  Confidence Scores:")
for sentiment, score in sentiment_response['SentimentScore'].items():
    bar = "█" * int(score * 20)  # Visual bar
    print(f"    {sentiment:10s}: {score:.3f} {bar}")

In [ ]:
# =============================================================================
# DEMO: ENTITY AND KEY PHRASE EXTRACTION
# =============================================================================
# Now let's extract entities and key phrases from the same text

# Detect named entities (people, places, organizations, dates, etc.)
entities_response = comprehend.detect_entities(
    Text=sample_text,
    LanguageCode='en'
)

print("Named Entities Found:")
print("-" * 50)
for entity in entities_response['Entities']:
    print(f"  {entity['Type']:15s} | {entity['Text']:30s} | Confidence: {entity['Score']:.2f}")

# Detect key phrases (important topics/concepts)
keyphrases_response = comprehend.detect_key_phrases(
    Text=sample_text,
    LanguageCode='en'
)

print("\n" + "=" * 60)
print("Key Phrases Found:")
print("-" * 50)
for phrase in keyphrases_response['KeyPhrases'][:10]:  # Show top 10
    print(f"  • {phrase['Text']} (confidence: {phrase['Score']:.2f})")

## Lab 1: Analyze All Customer Feedback with Comprehend

Now it's your turn! You'll write a function to analyze all 20 customer feedback records and collect the results into a DataFrame.

### Your Task

1. Complete the `analyze_feedback()` function that:
   - Takes a text string as input
   - Calls all three Comprehend APIs (sentiment, entities, key phrases)
   - Returns a dictionary with the results

2. Loop through all feedback records and collect results

3. Create a pandas DataFrame with the analysis results

### Expected Output

A DataFrame with columns:
- `feedback_id`: The original feedback ID
- `sentiment`: POSITIVE, NEGATIVE, NEUTRAL, or MIXED
- `positive_score`: Confidence score for positive sentiment
- `negative_score`: Confidence score for negative sentiment  
- `entity_count`: Number of entities found
- `entities`: List of entity texts
- `key_phrases`: Top 5 key phrases

### Hints

- Use `time.sleep(0.1)` between API calls to avoid throttling
- Access sentiment scores with `response['SentimentScore']['Positive']`
- Extract entity texts with list comprehension: `[e['Text'] for e in entities]`

In [ ]:
# =============================================================================
# LAB 1 SOLUTION: ANALYZE ALL FEEDBACK WITH COMPREHEND
# =============================================================================

def analyze_feedback(text):
    """
    Analyze a single feedback text with Comprehend.
    Returns a dictionary with sentiment, entities, and key phrases.
    """
    results = {}
    
    # 1. Detect Sentiment
    sentiment_response = comprehend.detect_sentiment(
        Text=text,
        LanguageCode='en'
    )
    
    # Extract sentiment and scores from response
    results['sentiment'] = sentiment_response['Sentiment']
    results['sentiment_scores'] = sentiment_response['SentimentScore']
    
    # 2. Detect Entities
    entities_response = comprehend.detect_entities(
        Text=text,
        LanguageCode='en'
    )
    
    # Extract entity texts as a list
    results['entities'] = entities_response['Entities']
    
    # 3. Detect Key Phrases
    keyphrases_response = comprehend.detect_key_phrases(
        Text=text,
        LanguageCode='en'
    )
    
    # Extract key phrase texts as a list
    results['key_phrases'] = keyphrases_response['KeyPhrases']
    
    return results


# Process all feedback records
print("Analyzing customer feedback with Comprehend...\n")

comprehend_results = []

for idx, row in feedback_df.iterrows():
    print(f"  Processing {row['feedback_id']}...", end=" ")
    
    # Call analyze_feedback() with the feedback text
    analysis = analyze_feedback(row['feedback_text'])
    
    # Extract the key information into a result dictionary
    result = {
        'feedback_id': row['feedback_id'],
        'customer_id': row['customer_id'],
        'sentiment': analysis['sentiment'],
        'positive_score': round(analysis['sentiment_scores']['Positive'], 3),
        'negative_score': round(analysis['sentiment_scores']['Negative'], 3),
        'entity_count': len(analysis['entities']),
        'entities': [e['Text'] for e in analysis['entities']],
        'key_phrases': [kp['Text'] for kp in analysis['key_phrases'][:5]]  # Top 5
    }
    
    comprehend_results.append(result)
    print(f"✓ {analysis['sentiment']}")
    
    # Small delay to avoid API throttling
    time.sleep(0.1)

# Create DataFrame from results
comprehend_df = pd.DataFrame(comprehend_results)
print(f"\n✅ Comprehend analysis complete!")
print(f"   Processed: {len(comprehend_df)} records")

In [ ]:
# =============================================================================
# VIEW COMPREHEND RESULTS
# =============================================================================

# Sentiment distribution
print("Sentiment Distribution:")
print(comprehend_df['sentiment'].value_counts())
print()

# View the DataFrame
print("Sample Results:")
display(comprehend_df.head())

# Detailed view of one analysis
print("\n" + "=" * 60)
print("Detailed Analysis - First Record:")
print("=" * 60)
sample = comprehend_df.iloc[0]
print(f"Feedback ID:    {sample['feedback_id']}")
print(f"Sentiment:      {sample['sentiment']}")
print(f"Positive Score: {sample['positive_score']}")
print(f"Negative Score: {sample['negative_score']}")
print(f"Entities:       {sample['entities']}")
print(f"Key Phrases:    {sample['key_phrases']}")

# Section 2: Amazon Textract - Document Processing

Amazon Textract extracts text, forms, and tables from scanned documents and images. It's like OCR (Optical Character Recognition) on steroids.

## What Can Textract Do?

| Feature | Purpose | Use Case |
|---------|---------|----------|
| **Text Detection** | Extract raw text (OCR) | Digitize printed documents |
| **Forms Extraction** | Extract key-value pairs | Process filled forms |
| **Table Extraction** | Extract tabular data | Convert tables to structured data |

## API Pattern

Textract analyzes documents stored in S3:

```python
response = textract.analyze_document(
    Document={
        'S3Object': {
            'Bucket': 'my-bucket',
            'Name': 'documents/form.png'
        }
    },
    FeatureTypes=['FORMS']  # or ['TABLES'] or both
)

# Response contains "Blocks" - each block is a piece of detected content
# BlockType can be: PAGE, LINE, WORD, KEY_VALUE_SET, TABLE, CELL
```

## Understanding Textract Blocks

Textract returns results as **blocks** with relationships:
- **KEY_VALUE_SET** blocks represent form fields (e.g., "Name: John Smith")
- Each KEY has a CHILD relationship to WORD blocks
- Each KEY has a VALUE relationship to its corresponding value

We need helper functions to parse these relationships.

In [ ]:
# =============================================================================
# HELPER FUNCTIONS FOR TEXTRACT RESPONSE PARSING
# =============================================================================
# These functions help us extract structured data from Textract's block format

def get_text_from_block(block, block_map):
    """
    Extract text from a block and its children.
    Blocks have relationships to CHILD blocks which contain the actual text.
    """
    text = ""
    if 'Relationships' in block:
        for rel in block['Relationships']:
            if rel['Type'] == 'CHILD':
                for child_id in rel['Ids']:
                    child = block_map.get(child_id, {})
                    if child.get('BlockType') == 'WORD':
                        text += child.get('Text', '') + ' '
                    elif child.get('BlockType') == 'SELECTION_ELEMENT':
                        # Checkboxes: SELECTED or NOT_SELECTED
                        if child.get('SelectionStatus') == 'SELECTED':
                            text += '[X] '
                        else:
                            text += '[ ] '
    return text.strip()


def get_key_value_pairs(blocks):
    """
    Extract key-value pairs from Textract response blocks.
    Example: "Name:" -> "John Smith"
    """
    # Create lookup maps
    key_map = {}
    value_map = {}
    block_map = {}
    
    # First pass: organize blocks by type
    for block in blocks:
        block_id = block['Id']
        block_map[block_id] = block
        
        if block['BlockType'] == 'KEY_VALUE_SET':
            if 'KEY' in block.get('EntityTypes', []):
                key_map[block_id] = block
            else:
                value_map[block_id] = block
    
    # Second pass: match keys to values
    kvs = {}
    for key_id, key_block in key_map.items():
        # Get the key text
        key_text = get_text_from_block(key_block, block_map)
        
        # Find the corresponding value
        value_text = ""
        if 'Relationships' in key_block:
            for rel in key_block['Relationships']:
                if rel['Type'] == 'VALUE':
                    for value_id in rel['Ids']:
                        if value_id in value_map:
                            value_text = get_text_from_block(value_map[value_id], block_map)
        
        if key_text:
            kvs[key_text.strip()] = value_text.strip()
    
    return kvs

print("✓ Helper functions defined")

In [ ]:
# =============================================================================
# DEMO: PROCESS A SINGLE FORM WITH TEXTRACT
# =============================================================================

# Get the first scanned form
sample_form_key = f"{DATA_PREFIX}/scanned_forms/form_001.png"

print(f"Processing: {sample_form_key.split('/')[-1]}")
print("=" * 60)

# Call Textract analyze_document with FORMS feature
response = textract.analyze_document(
    Document={
        'S3Object': {
            'Bucket': BUCKET_NAME,
            'Name': sample_form_key
        }
    },
    FeatureTypes=['FORMS']  # Extract key-value pairs from forms
)

print(f"Textract found {len(response['Blocks'])} blocks in the document\n")

# Use our helper function to extract key-value pairs
key_values = get_key_value_pairs(response['Blocks'])

print("Extracted Form Fields:")
print("-" * 60)
for key, value in key_values.items():
    print(f"  {key:30s} : {value}")

print(f"\n✓ Successfully extracted {len(key_values)} form fields")

## Lab 2: Process All Scanned Forms with Textract

Now process all 5 scanned forms and extract the form fields from each.

### Your Task

1. List all PNG files in the `scanned_forms/` folder
2. Loop through each form and call Textract's `analyze_document()` API
3. Use the `get_key_value_pairs()` helper function to extract fields
4. Store results in a list of dictionaries
5. Create a DataFrame with the extracted data

### Expected Output

A DataFrame where each row represents one form, with columns for:
- `form_file`: The filename (e.g., "form_001.png")
- `s3_key`: The full S3 key
- Individual form fields as columns (Customer Name, Date, Order Number, etc.)

### Hints

- Use `s3.list_objects_v2()` to find all PNG files
- Filter for files ending with `.png`
- The form fields vary by form type, so some columns may have NaN values

In [ ]:
# =============================================================================
# LAB 2 SOLUTION: PROCESS ALL SCANNED FORMS
# =============================================================================

# List all scanned form files
response = s3.list_objects_v2(
    Bucket=BUCKET_NAME,
    Prefix=f"{DATA_PREFIX}/scanned_forms/"
)

# Extract keys that end with '.png'
form_files = [
    obj['Key'] for obj in response.get('Contents', [])
    if obj['Key'].endswith('.png')
]

print(f"Found {len(form_files)} scanned forms to process\n")

textract_results = []

for form_key in form_files:
    form_name = form_key.split('/')[-1]
    print(f"  Processing {form_name}...", end=" ")
    
    # Call Textract analyze_document with FORMS feature
    response = textract.analyze_document(
        Document={
            'S3Object': {
                'Bucket': BUCKET_NAME,
                'Name': form_key
            }
        },
        FeatureTypes=['FORMS']  # Extract key-value pairs
    )
    
    # Extract key-value pairs using helper function
    key_values = get_key_value_pairs(response['Blocks'])
    
    # Store results
    result = {
        'form_file': form_name,
        's3_key': form_key,
    }
    # Add all key-value pairs to the result dictionary
    # This flattens the form fields into columns
    result.update(key_values)
    
    textract_results.append(result)
    print(f"✓ Found {len(key_values)} fields")

# Create DataFrame
textract_df = pd.DataFrame(textract_results)
print(f"\n✅ Textract processing complete!")
print(f"   Processed {len(textract_df)} forms")

# Section 3: Amazon Rekognition - Image Analysis

Amazon Rekognition detects objects, scenes, activities, and faces in images. It uses deep learning models trained on millions of images.

## What Can Rekognition Do?

| API | Purpose | Output |
|-----|---------|--------|
| `detect_labels()` | Identify objects, scenes, concepts | Labels with confidence scores (e.g., "Laptop", "Electronics") |
| `detect_faces()` | Detect faces and analyze attributes | Age range, emotions, facial features |
| `detect_text()` | Extract text from images | Text blocks with bounding boxes |

## API Pattern

```python
response = rekognition.detect_labels(
    Image={
        'S3Object': {
            'Bucket': 'my-bucket',
            'Name': 'images/photo.jpg'
        }
    },
    MaxLabels=10,          # Return top 10 labels
    MinConfidence=70.0     # Only labels with >70% confidence
)

# Access labels
for label in response['Labels']:
    print(f"{label['Name']}: {label['Confidence']:.1f}%")
```

## Demo: Analyzing Product Images

We'll detect objects and labels in product photos customers send.

In [ ]:
# =============================================================================
# DEMO: ANALYZE A SINGLE PRODUCT IMAGE
# =============================================================================

# Get the first product image
sample_image_key = f"{DATA_PREFIX}/product_images/product_001.jpg"

print(f"Analyzing: {sample_image_key.split('/')[-1]}")
print("=" * 60)

# Call Rekognition detect_labels
response = rekognition.detect_labels(
    Image={
        'S3Object': {
            'Bucket': BUCKET_NAME,
            'Name': sample_image_key
        }
    },
    MaxLabels=15,           # Return up to 15 labels
    MinConfidence=70.0      # Only return labels with >70% confidence
)

# Display results
labels = response['Labels']
print(f"\nRekognition detected {len(labels)} labels:\n")

for label in labels:
    # Create a visual confidence bar
    confidence = label['Confidence']
    bar = "█" * int(confidence / 5)  # Scale to 20 character width
    print(f"  {label['Name']:20s} {confidence:5.1f}% {bar}")

# Show primary label
primary_label = labels[0] if labels else None
if primary_label:
    print(f"\n✓ Primary object detected: {primary_label['Name']} ({primary_label['Confidence']:.1f}% confidence)")

In [ ]:
# =============================================================================
# LAB 3 SOLUTION: ANALYZE ALL PRODUCT IMAGES
# =============================================================================

# List product image files
response = s3.list_objects_v2(
    Bucket=BUCKET_NAME,
    Prefix=f"{DATA_PREFIX}/product_images/"
)

# Extract keys that end with '.jpg' or '.jpeg'
image_files = [
    obj['Key'] for obj in response.get('Contents', [])
    if obj['Key'].endswith('.jpg') or obj['Key'].endswith('.jpeg')
]

print(f"Found {len(image_files)} product images to analyze\n")

rekognition_results = []

for image_key in image_files:
    image_name = image_key.split('/')[-1]
    print(f"  Analyzing {image_name}...", end=" ")
    
    # Call Rekognition detect_labels
    response = rekognition.detect_labels(
        Image={
            'S3Object': {
                'Bucket': BUCKET_NAME,
                'Name': image_key
            }
        },
        MaxLabels=15,           # Return up to 15 labels
        MinConfidence=70.0      # Only return labels with >70% confidence
    )
    
    # Extract labels from response
    labels = response['Labels']
    
    result = {
        'image_file': image_name,
        's3_key': image_key,
        'labels': [l['Name'] for l in labels],  # List of label names
        'primary_label': labels[0]['Name'] if labels else None,  # First label is highest confidence
        'primary_confidence': round(labels[0]['Confidence'], 1) if labels else None,  # Rounded to 1 decimal
        'label_count': len(labels)
    }
    
    rekognition_results.append(result)
    print(f"✓ Found {len(labels)} labels")

# Create DataFrame
rekognition_df = pd.DataFrame(rekognition_results)
print(f"\n✅ Rekognition analysis complete!")
print(f"   Processed {len(rekognition_df)} images")

# Section 4: Save Results to S3

Now that we've extracted insights from all three data sources, let's save the structured results back to S3 for downstream systems to use.

## Why Save to S3?

- **Persistence**: Results are stored permanently, not just in memory
- **Downstream Systems**: Dashboards, ML models, and analytics tools can consume this data
- **Audit Trail**: Keep a record of what was extracted and when
- **Cost Effective**: S3 storage is cheap compared to database storage

## What We'll Save

1. **Comprehend results** (sentiment, entities, key phrases) → CSV
2. **Textract results** (extracted form fields) → CSV  
3. **Rekognition results** (detected labels) → CSV
4. **Summary metadata** (counts, timestamps) → JSON

In [ ]:
# =============================================================================
# SAVE ALL RESULTS TO S3
# =============================================================================

# Create timestamp for output files
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

print("Saving results to S3...")
print("=" * 60)

# 1. Save Comprehend results
comprehend_key = f"{OUTPUT_PREFIX}/comprehend_analysis_{timestamp}.csv"
csv_buffer = comprehend_df.to_csv(index=False)
s3.put_object(Bucket=BUCKET_NAME, Key=comprehend_key, Body=csv_buffer)
print(f"✓ Comprehend:   s3://{BUCKET_NAME}/{comprehend_key}")

# 2. Save Textract results
textract_key = f"{OUTPUT_PREFIX}/textract_forms_{timestamp}.csv"
csv_buffer = textract_df.to_csv(index=False)
s3.put_object(Bucket=BUCKET_NAME, Key=textract_key, Body=csv_buffer)
print(f"✓ Textract:     s3://{BUCKET_NAME}/{textract_key}")

# 3. Save Rekognition results (convert lists to strings for CSV)
rekognition_export = rekognition_df.copy()
rekognition_export['labels'] = rekognition_export['labels'].apply(lambda x: '|'.join(x) if isinstance(x, list) else x)
rekognition_key = f"{OUTPUT_PREFIX}/rekognition_labels_{timestamp}.csv"
csv_buffer = rekognition_export.to_csv(index=False)
s3.put_object(Bucket=BUCKET_NAME, Key=rekognition_key, Body=csv_buffer)
print(f"✓ Rekognition:  s3://{BUCKET_NAME}/{rekognition_key}")

# 4. Save summary metadata as JSON
# Note: Convert numpy int64 to Python int for JSON serialization
summary = {
    'extraction_timestamp': timestamp,
    'comprehend': {
        'records_processed': int(len(comprehend_df)),
        'sentiment_distribution': {k: int(v) for k, v in comprehend_df['sentiment'].value_counts().to_dict().items()} if 'sentiment' in comprehend_df.columns else {}
    },
    'textract': {
        'forms_processed': int(len(textract_df))
    },
    'rekognition': {
        'images_processed': int(len(rekognition_df)),
        'total_labels_detected': int(rekognition_df['label_count'].sum()) if 'label_count' in rekognition_df.columns else 0
    }
}

summary_key = f"{OUTPUT_PREFIX}/extraction_summary_{timestamp}.json"
s3.put_object(Bucket=BUCKET_NAME, Key=summary_key, Body=json.dumps(summary, indent=2))
print(f"✓ Summary:      s3://{BUCKET_NAME}/{summary_key}")

print("\n" + "=" * 60)
print("All data extracted and saved to S3!")
print("=" * 60)

# Summary

Congratulations! You've built a complete data extraction pipeline using AWS AI Services.

## What You Learned

| Service | Input | Output | Use Case |
|---------|-------|--------|----------|
| **Comprehend** | Text | Sentiment, Entities, Key Phrases | Customer feedback analysis |
| **Textract** | Images/PDFs | Text, Key-Value pairs | Form digitization |
| **Rekognition** | Images | Labels, Objects, Scenes | Product image classification |

## Key Takeaways

1. **boto3 clients** make it easy to call AWS services from Python
2. **No ML expertise required** - these are pre-trained models you can use immediately
3. **Structured output** - Convert unstructured data (text, images, documents) into structured DataFrames
4. **S3 integration** - Store results for downstream analytics and ML pipelines
5. **Pay per use** - You only pay for API calls, not for infrastructure

## The Pipeline You Built

```
Customer Feedback (3 channels)
    ↓
AWS AI Services (Comprehend, Textract, Rekognition)
    ↓
Structured Data (pandas DataFrames)
    ↓
S3 Storage (CSV + JSON)
    ↓
Ready for dashboards, ML models, analytics
```

## Next Week Preview

In Week 6, we'll build on this foundation:
- Use **Amazon Transcribe** to convert call center audio to text
- Chain AI services together (Transcribe → Comprehend → Translate)
- Build ML models on SageMaker using features from AI Services
- Run hyperparameter tuning jobs

# Optional / Extra Labs

If you finish early or want to explore more, try these advanced exercises:

## Extra Lab 1: Batch Processing with Comprehend

**Challenge**: Modify the Comprehend analysis to use batch APIs for better efficiency.

Comprehend offers batch APIs that process up to 25 documents per call:
- `batch_detect_sentiment()`
- `batch_detect_entities()`
- `batch_detect_key_phrases()`

**Task**: Refactor the comprehend analysis loop to process feedback in batches of 25.

**Hint**: Split the feedback_df into chunks of 25 rows, call the batch APIs, then combine results.

```python
# Example structure
def batch_analyze_feedback(texts):
    # texts is a list of up to 25 strings
    response = comprehend.batch_detect_sentiment(
        TextList=texts,
        LanguageCode='en'
    )
    # Process response['ResultList']
```

## Extra Lab 2: Table Extraction with Textract

**Challenge**: Extract tables from documents using Textract.

Textract can extract tabular data from documents with the `TABLES` feature.

**Task**:
1. Find or create a document with a table (invoice, expense report, etc.)
2. Upload to S3
3. Call Textract with `FeatureTypes=['TABLES']`
4. Parse the TABLE and CELL blocks to reconstruct the table
5. Convert to a pandas DataFrame

**Hint**: TABLE blocks have CHILD relationships to CELL blocks, which have row/column indices.

## Extra Lab 3: Face Detection with Rekognition

**Challenge**: Use Rekognition to detect faces and analyze attributes.

**Task**:
1. Find an image with faces (use a public dataset or create one)
2. Call `rekognition.detect_faces()` with `Attributes=['ALL']`
3. Extract age range, emotions, and facial features
4. Visualize the results

```python
response = rekognition.detect_faces(
    Image={'S3Object': {'Bucket': BUCKET_NAME, 'Name': 'image_key'}},
    Attributes=['ALL']  # Returns age, emotions, beard, eyeglasses, etc.
)

for face in response['FaceDetails']:
    print(f"Age: {face['AgeRange']}")
    print(f"Emotions: {face['Emotions']}")
```

## Extra Lab 4: Custom Comprehend Entity Recognition

**Challenge**: Train a custom entity recognizer for domain-specific entities.

If you have domain-specific entities (product codes, internal IDs, etc.), you can train a custom Comprehend model.

**Task**: Research Amazon Comprehend Custom Entity Recognition and outline the steps needed to train a custom model for your use case.

## Resources

- [Amazon Comprehend Documentation](https://docs.aws.amazon.com/comprehend/)
- [Amazon Textract Documentation](https://docs.aws.amazon.com/textract/)
- [Amazon Rekognition Documentation](https://docs.aws.amazon.com/rekognition/)
- [AWS SDK for Python (boto3)](https://boto3.amazonaws.com/v1/documentation/api/latest/index.html)

# Well Done!

You've successfully completed Week 5 of the AI for Data Scientists Academy.

## Skills Acquired

- **AWS Environment Navigation**: Working with SageMaker notebooks, boto3 clients, and S3
- **NLP with Comprehend**: Sentiment analysis, named entity recognition, key phrase extraction
- **Document Processing with Textract**: OCR, form field extraction, key-value pair parsing
- **Computer Vision with Rekognition**: Object detection, label classification
- **Data Pipeline Building**: End-to-end extraction, transformation, and storage

## What's Next

Next week, you'll level up by:
1. Processing audio data with Amazon Transcribe
2. Chaining multiple AI services together
3. Using AI Service outputs as features for ML models
4. Training your first SageMaker model

Keep your SageMaker notebook environment ready - we'll build on what you learned today!

**See you in Week 6!**

In [ ]:
# =============================================================================
# VIEW REKOGNITION RESULTS
# =============================================================================

print("Product Image Analysis Results:\n")
display(rekognition_df)

# Show detailed labels for each image
print("\n" + "=" * 60)
print("Detailed Labels Per Image:")
print("=" * 60)
for idx, row in rekognition_df.iterrows():
    print(f"\n{row['image_file']}:")
    print(f"  Primary: {row['primary_label']} ({row['primary_confidence']}% confidence)")
    print(f"  All labels: {', '.join(row['labels'][:5])}...")  # Show first 5

## Lab 3: Analyze All Product Images with Rekognition

Process all 6 product images and collect the detected labels.

### Your Task

1. List all JPG files in the `product_images/` folder
2. Loop through each image and call `detect_labels()`
3. Store results including:
   - Image filename
   - List of all detected labels
   - Primary (top) label and its confidence
   - Total label count
4. Create a DataFrame with the results

### Expected Output

A DataFrame with columns:
- `image_file`: The filename
- `s3_key`: Full S3 key
- `labels`: List of all label names
- `primary_label`: The top label
- `primary_confidence`: Confidence of primary label
- `label_count`: Total number of labels detected

### Hints

- Use `MinConfidence=70.0` to filter low-confidence labels
- Access label name with `label['Name']`
- The first label in the list is the primary (highest confidence) label

In [ ]:
# =============================================================================
# VIEW TEXTRACT RESULTS
# =============================================================================

print("Extracted Form Fields:\n")

# Display the structured data
display(textract_df)

# Show detailed fields from first form
print("\n" + "=" * 60)
print("Detailed Fields - First Form:")
print("=" * 60)
first_form = textract_df.iloc[0]
for key, value in first_form.items():
    if pd.notna(value):
        print(f"  {key:25s} : {value}")